# Qwen3.5 agent: development smoke evaluation

Uses a local HTTP twin and the Certify development split. The default is
NF4, greedy decoding, 1,024 output tokens and 12 turns. A completed submission
does not establish action benefit or safety. Dataset-derived inputs and run
archives remain private. The package contains no model weights or raw Bosch data.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys, uuid, zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/smartbai_agent_smoke')
WORK.mkdir(parents=True, exist_ok=True)
roots = sorted({p.parent for p in INPUT.rglob('agent_smoke_bundle_manifest.json')
                if (p.parent / 'llm_adapter/agent.py').is_file()})
if not roots:
    archives = sorted(INPUT.rglob('smartbai_agent_smoke_bundle.zip'))
    if len(archives) != 1:
        raise RuntimeError('Attach exactly one current private agent bundle.')
    destination = WORK / 'bundle_source'
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    roots = sorted({p.parent for p in destination.rglob('agent_smoke_bundle_manifest.json')
                    if (p.parent / 'llm_adapter/agent.py').is_file()})
if len(roots) != 1:
    raise RuntimeError('Expected one project bundle; remove duplicate inputs.')
source = roots[0]
manifest = json.loads((source / 'agent_smoke_bundle_manifest.json').read_text())
if manifest.get('format_version') != 1:
    raise RuntimeError('Use the current package generated by build_bundle.py.')
for relative, expected in manifest['files_sha256'].items():
    path = (source / relative).resolve()
    if not path.is_relative_to(source.resolve()):
        raise ValueError('Unsafe manifest path')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('Hash mismatch: ' + relative)
PROJECT = WORK / 'project'
shutil.copytree(source, PROJECT, dirs_exist_ok=True)
ART = PROJECT / 'experiments/input_rebuild/artifacts'
print('Verified package:', source)


In [ ]:
if 'transformers' in sys.modules or 'huggingface_hub' in sys.modules:
    raise RuntimeError('Restart the kernel before installing the inference environment.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(PROJECT / 'llm_adapter/requirements_qwen35_nf4.txt'),
                'numpy', 'pandas', 'scipy'], check=True)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('This evaluation requires CUDA; select two T4 GPUs.')
print([{'gpu': i, 'name': torch.cuda.get_device_name(i),
        'free_GiB': round(torch.cuda.mem_get_info(i)[0] / 2**30, 2)}
       for i in range(torch.cuda.device_count())])
MODE = 'certify'
CASE_LIMIT = 1
MAX_TURNS = 12
MAX_NEW_TOKENS = 1024
NF4 = True
sys.path.insert(0, str(PROJECT))
from llm_adapter.development import run_development
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
RUN_DIR = WORK / 'runs' / RUN_ID
print('Output:', RUN_DIR)


In [ ]:
fixture = run_development(
    repo=PROJECT, artifacts=ART, out=WORK / 'fixtures' / RUN_ID,
    mode=MODE, backend_name='scripted', limit=1, max_turns=MAX_TURNS,
)
assert fixture['finished'] == 1
print('Scripted HTTP fixture only:', fixture)


In [ ]:
result = run_development(
    repo=PROJECT, artifacts=ART, out=RUN_DIR, mode=MODE,
    backend_name='qwen35', limit=CASE_LIMIT, max_turns=MAX_TURNS,
    nf4=NF4, max_new_tokens=MAX_NEW_TOKENS,
    resume=(RUN_DIR / 'run_manifest.json').exists(),
)
print(json.dumps(result, indent=2))


In [ ]:
from IPython.display import FileLink, display
if not RUN_DIR.exists():
    raise RuntimeError('No run output was created; inspect the setup error.')
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Private run archive:', archive)
display(FileLink(archive))
